# S6E8 v2 — méta-stack logistique honnête (bibliothèque OOF 47 modèles + Naji + notre v1)

Playbook S6E7 : la victoire est dans le méta-ensemble, pas le single model.
Méthode inspirée de riponce (stack logit + régimes missingness, OOF 0.96950) et rayk (percentile-rank
conditionné par missingness, LB 0.97069), reconstruite et validée par folds identiques.
Trois candidats comparés honnêtement : A) stack global, B) stack + régimes missingness, C) blend percentile
conditionné. On soumet le meilleur en CV fold-consistant — une seule soumission.

In [ ]:
import glob, os, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata, spearmanr

warnings.filterwarnings('ignore')
TARGET = 'addicted_label'; ID_COL = 'id'
SEED = 42; N_FOLDS = 5

CAND = ['/kaggle/input/competitions/playground-series-s6e8',
        '/kaggle/input/playground-series-s6e8', './data', '.']
DATA_DIR = next(p for p in CAND if os.path.exists(os.path.join(p, 'train.csv')))
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sample = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
y = train[TARGET].to_numpy()
feat_cols = [c for c in test.columns if c != ID_COL]
miss_train = train[feat_cols].isna().sum(axis=1).to_numpy()
miss_test = test[feat_cols].isna().sum(axis=1).to_numpy()
print('train/test:', train.shape, test.shape)

In [ ]:
# ---------- 1) Chargement de la bibliothèque OOF ----------
def collect_pairs():
    pairs = {}
    for op in glob.glob('/kaggle/input/**/oof_*.npy', recursive=True):
        name = os.path.basename(op)[4:-4]
        tp = os.path.join(os.path.dirname(op), f'test_{name}.npy')
        if os.path.exists(tp):
            pairs[name] = (op, tp)
    return pairs

pairs = collect_pairs()
print(f'{len(pairs)} paires oof/test trouvées')

O_list, T_list, names = [], [], []
for name, (op, tp) in sorted(pairs.items()):
    o = np.load(op).astype(np.float64); t = np.load(tp).astype(np.float64)
    if o.shape != (len(train),) or t.shape != (len(test),):
        print('skip shape:', name); continue
    if not (np.isfinite(o).all() and np.isfinite(t).all()):
        print('skip non-finite:', name); continue
    O_list.append(o); T_list.append(t); names.append(name)

# membres externes : Naji 10_blend + notre v1
for tag, pat_o, pat_t in [
    ('naji10', '**/10_blend_oof*.csv', '**/10_blend_sub*.csv'),
    ('ours_v1', '**/oof.npy', '**/test_proba.npy')]:
    ho = [p for p in glob.glob('/kaggle/input/' + pat_o, recursive=True)]
    ht = [p for p in glob.glob('/kaggle/input/' + pat_t, recursive=True)]
    if ho and ht:
        if ho[0].endswith('.csv'):
            o = pd.read_csv(ho[0]); t = pd.read_csv(ht[0])
            co = [c for c in o.columns if c != ID_COL][-1]; ct = [c for c in t.columns if c != ID_COL][-1]
            ov, tv = o[co].to_numpy(float), t[ct].to_numpy(float)
        else:
            ov, tv = np.load(ho[0]).astype(float), np.load(ht[0]).astype(float)
        if ov.shape == (len(train),) and tv.shape == (len(test),) and np.isfinite(ov).all() and np.isfinite(tv).all():
            O_list.append(ov); T_list.append(tv); names.append(tag)
            print('membre externe ajouté:', tag, '| AUC oof =', round(roc_auc_score(y, ov), 5))

O = np.column_stack(O_list); Tm = np.column_stack(T_list)
print('membres chargés:', len(names))
aucs = [roc_auc_score(y, O[:, i]) for i in range(O.shape[1])]
print('AUC membres: min %.5f | médiane %.5f | max %.5f' % (min(aucs), np.median(aucs), max(aucs)))

# déduplication (corr > 0.9995) sur sous-échantillon
rng = np.random.RandomState(SEED)
sub = rng.choice(len(y), 100_000, replace=False)
C = np.corrcoef(O[sub], rowvar=False)
drop = set()
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        if C[i, j] > 0.9995 and j not in drop:
            drop.add(j)
keep = [i for i in range(len(names)) if i not in drop]
print('dédoublonnage:', len(names) - len(keep), 'retirés ->', len(keep), 'conservés')
O = O[:, keep]; Tm = Tm[:, keep]; names = [names[i] for i in keep]

In [ ]:
# ---------- 2) Espace logit + features de régime ----------
def logit(p):
    return np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))

OL = logit(O); TL = logit(Tm)

def regime_feats(L, miss):
    mu = L.mean(1); sd = L.std(1)  # disagreement = écart-type inter-membres
    complete = (miss == 0).astype(float); hi = (miss >= 4).astype(float)
    return np.column_stack([L, mu, sd, mu * complete, mu * hi, sd * complete, sd * hi])

XA = OL; XA_t = TL                                  # A: global
XB = regime_feats(OL, miss_train); XB_t = regime_feats(TL, miss_test)  # B: + régimes
print('XA', XA.shape, '| XB', XB.shape)

In [ ]:
# ---------- 3) CV honnête : A vs B, plusieurs C ----------
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(XA, y))

def cv_stack(X, C=1.0):
    oof = np.zeros(len(y))
    aucs = []
    for tr, va in folds:
        lr = LogisticRegression(C=C, max_iter=2000, n_jobs=-1)
        lr.fit(X[tr], y[tr])
        oof[va] = lr.predict_proba(X[va])[:, 1]
        aucs.append(roc_auc_score(y[va], oof[va]))
    return oof, np.mean(aucs), np.std(aucs)

results = {}
for label, X in [('A global', XA), ('B regimes', XB)]:
    for C in [0.1, 1.0, 10.0]:
        _, m, s = cv_stack(X, C)
        results[(label, C)] = (m, s)
        print(f'{label} C={C}: AUC {m:.6f} ± {s:.6f}', flush=True)

best_key = max(results, key=lambda k: results[k][0])
print('meilleur:', best_key, results[best_key])

In [ ]:
# ---------- 4) Candidat C : blend percentile conditionné (style rayk) ----------
def pct_rank(x):
    return rankdata(x) / len(x)

oof_A, _, _ = cv_stack(XA, 1.0)
naji_idx = names.index('naji10') if 'naji10' in names else None
if naji_idx is not None:
    oof_naji = O[:, naji_idx]; tst_naji = Tm[:, naji_idx]
    r = spearmanr(oof_A, oof_naji).statistic
    print(f'corr spearman stack/naji: {r:.4f}')
    oofC = np.where(miss_train == 0, 0.9 * pct_rank(oof_A) + 0.1 * pct_rank(oof_naji),
                    0.6 * pct_rank(oof_A) + 0.4 * pct_rank(oof_naji))
    auc_folds = [roc_auc_score(y[va], oofC[va]) for _, va in folds]
    print('C percentile-blend: AUC %.6f ± %.6f' % (np.mean(auc_folds), np.std(auc_folds)))
    auc_A_folds = [roc_auc_score(y[va], oof_A[va]) for _, va in folds]
    print('(rappel A:        AUC %.6f ± %.6f)' % (np.mean(auc_A_folds), np.std(auc_A_folds)))

In [ ]:
# ---------- 5) Entraînement final full-data + soumission du meilleur candidat ----------
label, C = best_key
Xf, Xf_t = (XA, XA_t) if label == 'A global' else (XB, XB_t)
oof_best, m_oof, s_oof = cv_stack(Xf, C)
lr_full = LogisticRegression(C=C, max_iter=2000, n_jobs=-1).fit(Xf, y)
tst_stack = lr_full.predict_proba(Xf_t)[:, 1]

# comparer avec C si disponible
use_C = False
if naji_idx is not None:
    oofC = np.where(miss_train == 0, 0.9 * pct_rank(oof_A) + 0.1 * pct_rank(oof_naji),
                    0.6 * pct_rank(oof_A) + 0.4 * pct_rank(oof_naji))
    tstC = np.where(miss_test == 0, 0.9 * pct_rank(tst_stack) + 0.1 * pct_rank(tst_naji),
                    0.6 * pct_rank(tst_stack) + 0.4 * pct_rank(tst_naji))
    aucC = np.mean([roc_auc_score(y[va], oofC[va]) for _, va in folds])
    if aucC > m_oof:
        use_C = True
        print(f'candidat C retenu (CV {aucC:.6f} > stack {m_oof:.6f})')

final_test = tstC if use_C else tst_stack
final_oof = oofC if use_C else oof_best
np.save('oof_stack.npy', final_oof); np.save('test_stack.npy', final_test)
sub = sample.copy(); sub[TARGET] = final_test
sub.to_csv('submission.csv', index=False)
print('soumission écrite | OOF CV =', round(float(np.mean([roc_auc_score(y[va], final_oof[va]) for _, va in folds])), 6))
print(sub[TARGET].describe())